# Historical data: 32 × 32 EPI, navigators and a B0 map

For the current experimental workflow, open **02_acdc_experiment_step_by_step.ipynb**. This historical notebook retains the original two EPI scans and the DICOM-based B0 reconstruction.
The stages are visible: raw data → readout polarities → navigators → Fourier reconstruction → multi-echo GRE → B0 → MATLAB export.

**Terminology:** “baseline” means without an added A22 perturbation. This scan is not a measurement at zero B0.
A22 is a scanner shim term set to 1000 µT/m², rather than an ACDC channel.
The six DICOM files contain magnitude and phase for **three echoes of one perturbed state**.

The phantom was scanned without ACDC coils. The separately supplied 3 T responses support a **virtual** optimization of the measured B0; these coils were absent during acquisition. The initial digital twin is conditional because the complete pair of baseline and perturbed B0 maps was not measured.

The historical nearest-neighbour extension below is an explicit simulation assumption. The new experimental notebook retains missing B0 values and requires measured coverage of the selected ROI.


In [ ]:
from pathlib import Path
import contextlib, io, json, hashlib, inspect
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage
from scipy.io import savemat
import pydicom
import twixtools
import pypulseq as pp
from skimage.restoration import unwrap_phase
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / 'scan_twix').is_dir():
    ROOT = ROOT / 'Experiment'
assert (ROOT / 'scan_twix').is_dir(), 'Run from Experiment or the project directory.'
FIG = ROOT / 'figures'; FIG.mkdir(exist_ok=True)
PREPARED = ROOT / 'prepared'; PREPARED.mkdir(exist_ok=True)
SEQUENCE_FILE = ROOT / 'sequence' / 'gre_epi_32x32_fov200mm_nav3_mintr.seq'
N = 32
FOV_M = 0.2

def ifftc(a, axes):
    return np.fft.fftshift(np.fft.ifftn(np.fft.ifftshift(a, axes=axes), axes=axes), axes=axes)

def fftc(a, axes):
    return np.fft.fftshift(np.fft.fftn(np.fft.ifftshift(a, axes=axes), axes=axes), axes=axes)

def save_figure(fig, name):
    fig.savefig(FIG / name, dpi=170, bbox_inches='tight')
    plt.show()

## 1. Check what was actually acquired

The final measurement block of each Twix contains the EPI; the preceding `AdjCoilSens` block is not an ACDC field calibration.
The `REFLECT` flag indicates reversed readout: these lines are reversed to place all samples in the same kx order.
The data contain **32 samples**, rather than 64; no arbitrary oversampling removal is applied.
The following table contains technical information only, without personal identifiers.


In [ ]:
def read_epi(path):
    # Some geometry helpers assume readout oversampling.
    # Here the MDH and the samples actually present are used.
    with contextlib.redirect_stdout(io.StringIO()):
        scan = twixtools.read_twix(str(path), verbose=False)[-1]
    image_blocks = sorted([m for m in scan['mdb'] if m.is_image_scan()], key=lambda m: m.cLin)
    nav_blocks = [m for m in scan['mdb'] if m.is_flag_set('PHASCOR')]
    assert len(image_blocks) == 32 and len(nav_blocks) == 3
    assert [m.cLin for m in image_blocks] == list(range(32))
    assert all(m.data.shape == (32, 32) for m in image_blocks + nav_blocks)
    # Arrays: [phase line, receive channel, readout sample].
    def ordered(m):
        a = np.asarray(m.data, dtype=np.complex128)
        return a[:, ::-1] if m.is_flag_set('REFLECT') else a
    kspace = np.stack([ordered(m) for m in image_blocks])
    nav = np.stack([ordered(m) for m in nav_blocks])
    negative = np.array([m.is_flag_set('REFLECT') for m in image_blocks])
    nav_negative = np.array([m.is_flag_set('REFLECT') for m in nav_blocks])
    assert np.array_equal(nav_negative, [False, True, False])
    hdr = scan['hdr']['MeasYaps']
    metadata = {
        'file': path.name, 'image_shape': list(kspace.shape), 'navigators': len(nav_blocks),
        'TE_s_header': float(hdr['alTE'][0]) * 1e-6,
        'TR_s_header': float(hdr['alTR'][0]) * 1e-6,
        'flip_deg_header': float(hdr['adFlipAngleDegree'][0]),
        'dwell_s': float(hdr['sRXSPEC']['alDwellTime'][0]) * 1e-9,
        'transmitter_Hz': int(hdr['sTXSPEC']['asNucleusInfo'][0]['lFrequency']),
        'matrix': [32, 32], 'FOV_m': FOV_M,
    }
    return {'kspace': kspace, 'nav': nav, 'negative': negative, 'metadata': metadata}

files = sorted((ROOT / 'scan_twix').glob('*.dat'))
assert len(files) == 2
baseline_file = next(p for p in files if 'A22_1000' not in p.name)
perturbed_file = next(p for p in files if 'A22_1000' in p.name)
scans = {'baseline': read_epi(baseline_file), 'perturbed': read_epi(perturbed_file)}
for name, scan in scans.items():
    print(name, json.dumps(scan['metadata'], ensure_ascii=False, indent=2))

## 2. Correct odd/even phase differences using three navigators

After a 1D readout transform, the negative navigator is compared with the average of the two positive navigators.
This symmetric placement reduces the effect of a common temporal phase evolution.
For each receive channel, a line $a_c+b_c x$ is fitted to the phase difference, weighted by signal strength.
Negative lines are multiplied by $\exp[-i(a_c+b_c x)]$ and transformed back to k-space.

This is an affine three-navigator correction, rather than a reproduction of the full proprietary Siemens pipeline.
Weak channels do not determine the correction of stronger channels.


In [ ]:
def correct_navigators(scan):
    hybrid_nav = ifftc(scan['nav'], axes=(2,))
    positive_reference = (hybrid_nav[0] + hybrid_nav[2]) / 2
    negative_nav = hybrid_nav[1]
    cross = negative_nav * np.conj(positive_reference)
    x = np.linspace(-1, 1, N)
    correction = np.zeros((32, N))
    coefficients = np.zeros((32, 2))
    for c in range(32):
        weight = np.abs(negative_nav[c]) * np.abs(positive_reference[c])
        good = weight > 0.10 * weight.max()
        if good.sum() < 4:
            continue
        phase = np.unwrap(np.angle(cross[c]))
        slope, intercept = np.polyfit(x[good], phase[good], 1, w=np.sqrt(weight[good]))
        correction[c] = intercept + slope*x
        coefficients[c] = [intercept, slope]
    hybrid_image = ifftc(scan['kspace'], axes=(2,))
    hybrid_image[scan['negative']] *= np.exp(-1j*correction)[None, :, :]
    corrected = fftc(hybrid_image, axes=(2,))
    coil_images = ifftc(corrected, axes=(0, 2))
    uncorrected_coils = ifftc(scan['kspace'], axes=(0, 2))
    # RSS = root sum of squares of the 32 receive-channel magnitudes.
    rss = np.sqrt(np.sum(np.abs(coil_images)**2, axis=1))
    rss_before = np.sqrt(np.sum(np.abs(uncorrected_coils)**2, axis=1))
    return {'corrected_kspace': corrected, 'coil_images': coil_images, 'rss': rss,
            'rss_before': rss_before, 'navigator_coefficients': coefficients}

recons = {name: correct_navigators(scan) for name, scan in scans.items()}
assert all(r['rss'].shape == (32, 32) for r in recons.values())
fig, axes = plt.subplots(2, 2, figsize=(10, 8), layout='constrained')
upper = max(r['rss_before'].max() for r in recons.values())
upper = max(upper, max(r['rss'].max() for r in recons.values()))
for row, (name, title) in enumerate([('baseline', 'No added A22 perturbation'), ('perturbed', 'With A22 perturbation')]):
    axes[row, 0].imshow(recons[name]['rss_before'], cmap='gray', vmin=0, vmax=upper)
    axes[row, 1].imshow(recons[name]['rss'], cmap='gray', vmin=0, vmax=upper)
    axes[row, 0].set_title(title + '\nBefore correction')
    axes[row, 1].set_title(title + '\nAfter navigator correction')
    for ax in axes[row]: ax.axis('off')
fig.suptitle('32 × 32 EPI: Nyquist ghost correction, shared scale')
save_figure(fig, 'epi_navigator_correction.png')

In [ ]:
# Check: remove a known affine error while preserving the ideal signal.
x = np.linspace(-1,1,N)
profile = np.exp(-3*x*x)
coil_profile = np.arange(1,33)[:,None]*profile[None,:]
phi_known = 0.35 + 0.50*x
nav_synthetic = fftc(np.stack([coil_profile,coil_profile*np.exp(1j*phi_known),coil_profile]), axes=(2,))
object_synthetic = np.exp(-3*(x[:,None]**2+x[None,:]**2))[:,None,:]*np.arange(1,33)[None,:,None]
k_ideal = fftc(object_synthetic,axes=(0,2))
negative_synthetic = np.arange(N)%2==0
hybrid_corrupted = ifftc(k_ideal,axes=(2,))
hybrid_corrupted[negative_synthetic] *= np.exp(1j*phi_known)[None,None,:]
k_corrupted = fftc(hybrid_corrupted,axes=(2,))
r_test = correct_navigators({'nav':nav_synthetic,'kspace':k_corrupted,'negative':negative_synthetic})
nav_error = np.linalg.norm(r_test['corrected_kspace']-k_ideal)/np.linalg.norm(k_ideal)
assert nav_error < 1e-12
print('Navigator correction test relative error:',nav_error)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), layout='constrained')
upper = max(r['rss'].max() for r in recons.values())
for ax, name, title in zip(axes, ['baseline', 'perturbed'], ['No added A22\nIntrinsic B0 unknown', 'With added A22']):
    ax.imshow(recons[name]['rss'], cmap='gray', vmin=0, vmax=upper)
    ax.set_title(title); ax.axis('off')
fig.suptitle('Measured EPI scans: 32 × 32, RSS combination')
save_figure(fig, 'epi_scans_32x32.png')

# fig, ax = plt.subplots(figsize=(8, 3))
# for name, recon in recons.items():
#     ax.plot(np.median(recon['navigator_coefficients'], axis=0), 'o-', label=name)
# ax.set_xticks([0, 1], ['Constant phase [rad]', 'Slope over normalized x [rad]'])
# ax.set_title('Correction parameters: median over channels'); ax.legend(); ax.grid(alpha=.2)
# save_figure(fig, 'navigator_coefficients.png')

## 3. Read the three GRE echoes and convert phase to radians

Echoes are paired by TE, rather than an assumed file order.
Enhanced DICOM stores some parameters in functional groups.
The Siemens phase conversion uses the rescale values present in these files:

$\phi = (2\,\mathrm{pixel}-4096)\,\pi/4096$.

TE values are converted from DICOM milliseconds to seconds before calculating B0.


In [ ]:
def dicom_value(ds, keyword):
    for element in ds.iterall():
        if element.keyword == keyword:
            return element.value
    raise KeyError(keyword)

mag_by_te, phase_by_te = {}, {}
gre_geometry = None
for path in sorted((ROOT / 'scan_dcm').rglob('*.dcm')):
    ds = pydicom.dcmread(path)
    te = float(dicom_value(ds, 'EffectiveEchoTime')) * 1e-3
    pixels = ds.pixel_array.astype(float).squeeze()
    assert pixels.shape == (128, 128)
    slope = float(dicom_value(ds, 'RescaleSlope'))
    intercept = float(dicom_value(ds, 'RescaleIntercept'))
    if 'P' in ds.ImageType:
        assert slope == 2 and intercept == -4096, 'Check this scanner phase scaling.'
        phase_by_te[te] = (pixels*slope + intercept) * np.pi/4096
    elif 'M' in ds.ImageType:
        mag_by_te[te] = pixels*slope + intercept
    this_geometry = {
        'position_mm': np.asarray(dicom_value(ds, 'ImagePositionPatient'), float),
        'orientation': np.asarray(dicom_value(ds, 'ImageOrientationPatient'), float),
        'spacing_mm': np.asarray(dicom_value(ds, 'PixelSpacing'), float),
    }
    if gre_geometry is not None:
        assert all(np.allclose(this_geometry[k], gre_geometry[k]) for k in gre_geometry)
    gre_geometry = this_geometry
TE_GRE = np.array(sorted(mag_by_te))
assert len(TE_GRE) == 3 and set(mag_by_te) == set(phase_by_te)
assert np.allclose(np.diff(TE_GRE), np.diff(TE_GRE)[0])
magnitude = np.stack([mag_by_te[t] for t in TE_GRE], axis=-1)
phase = np.stack([phase_by_te[t] for t in TE_GRE], axis=-1)
print('GRE TE [ms]:', TE_GRE*1000)
print('GRE grid:', magnitude.shape[:2], '| spacing [mm]:', gre_geometry['spacing_mm'])
fig, axes = plt.subplots(2, 3, figsize=(11, 7))
for e, te in enumerate(TE_GRE):
    axes[0, e].imshow(magnitude[..., e], cmap='gray', vmin=0, vmax=magnitude.max())
    axes[1, e].imshow(phase[..., e], cmap='twilight', vmin=-np.pi, vmax=np.pi)
    axes[0, e].set_title(f'Magnitude, TE={te*1000:.3f} ms')
    axes[1, e].set_title('Phase [rad]')
    for ax in axes[:, e]: ax.axis('off')
save_figure(fig, 'gre_magnitude_phase_3echoes.png')

## 4. Calculate B0 and its reliability mask

For two echoes, $B_0=\Delta\phi/(2\pi\Delta TE)$ in Hz.
The two successive increments are combined with a circular mean.
The residual $\arg\exp[i(\phi_3-2\phi_2+\phi_1)]$ checks consistency with linear phase evolution.

The mask retains pixels with magnitude of at least **5 DICOM units at all three echoes** and residual below **0.5 rad**.
Components smaller than 10 pixels are removed, without automatically filling dark holes.
These thresholds are visible and adjustable; they are not universal quality thresholds.

Spatial unwrapping resolves local discontinuities but may leave the absolute frequency branch unknown.
With ΔTE = 1.82 ms, adjacent branches differ by about 549.45 Hz. An independent reference is needed for a certain absolute offset.
The sign follows the exported phase differences; its relationship to the physical simulator convention must be checked using a field with known sign.

This historical notebook also makes an explicitly hypothetical nearest-neighbour extension for its virtual digital twin. That extension is not used in the current experimental notebook.


In [ ]:
MIN_MAGNITUDE = 5.0
MAX_PHASE_RESIDUAL = 0.5
MIN_COMPONENT_PIXELS = 10
B0_SIGN = +1.0  # Convention Δphase/(2πΔTE); change only after an independent sign check.

def wrapped(a):
    return np.angle(np.exp(1j*a))

delta12 = wrapped(phase[..., 1]-phase[..., 0])
delta23 = wrapped(phase[..., 2]-phase[..., 1])
phase_residual = np.abs(wrapped(delta23-delta12))
wrapped_increment = np.angle(np.exp(1j*delta12)+np.exp(1j*delta23))
quality = (magnitude.min(axis=-1) >= MIN_MAGNITUDE) & (phase_residual <= MAX_PHASE_RESIDUAL)
labels, nlabels = ndimage.label(quality)
sizes = np.bincount(labels.ravel())
keep_labels = np.flatnonzero(sizes >= MIN_COMPONENT_PIXELS)
keep_labels = keep_labels[keep_labels != 0]
quality = np.isin(labels, keep_labels)
assert quality.sum() > 20, 'Too few reliable points: inspect GRE signal and thresholds.'
masked_increment = np.ma.array(wrapped_increment, mask=~quality)
unwrap_options = {'rng': 0} if 'rng' in inspect.signature(unwrap_phase).parameters else {'seed': 0}
unwrapped_increment = unwrap_phase(masked_increment, **unwrap_options)
delta_te = TE_GRE[1]-TE_GRE[0]
b0_measured = np.where(quality, B0_SIGN*np.ma.getdata(unwrapped_increment)/(2*np.pi*delta_te), np.nan)
b0_wrapped = B0_SIGN*wrapped_increment/(2*np.pi*delta_te)
# Historical nearest-neighbour extension for a finite simulator: NOT a measurement.
nearest_indices = ndimage.distance_transform_edt(~quality, return_distances=False, return_indices=True)
b0_for_simulation = b0_measured[tuple(nearest_indices)]
assert np.isfinite(b0_for_simulation).all()
print('Reliable pixels:', int(quality.sum()), '/', quality.size)
print('Measured B0, percentiles 1/50/99 [Hz]:', np.nanpercentile(b0_measured, [1,50,99]))
print('Median disagreement between increments [Hz]:', np.median(phase_residual[quality])/(2*np.pi*delta_te))

limit = max(50, float(np.nanpercentile(np.abs(b0_measured), 98)))
fig, axes = plt.subplots(2, 2, figsize=(11, 9), layout='constrained')
plots = [(magnitude[..., 0], 'GRE magnitude, echo 1', 'gray', None),
         (phase_residual, 'Phase residual [rad]', 'magma', (0, np.pi)),
         (b0_measured, 'Measured B0 [Hz]\nReliability mask', 'coolwarm', (-limit,limit)),
         (b0_for_simulation, 'Extended B0 [Hz]\nNearest-neighbour assumption', 'coolwarm', (-limit,limit))]
for ax, (array, title, cmap, bounds) in zip(axes.ravel(), plots):
    kwargs = {} if bounds is None else {'vmin':bounds[0], 'vmax':bounds[1]}
    im = ax.imshow(array, cmap=cmap, **kwargs); ax.set_title(title); ax.axis('off')
    fig.colorbar(im, ax=ax, shrink=.8)
axes[0,0].contour(quality, levels=[.5], colors='lime', linewidths=.5)
fig.suptitle('Perturbed-acquisition B0: measured field and extension assumption')
save_figure(fig, 'b0_map_quality.png')

## 5. Extract the Pulseq ADC trajectory and timing

The `.seq` was found in `scanner/sequence` and copied here for reproducibility.
Gradients are integrated with PyPulseq, then the three navigator lines are removed.
Times are measured from the RF centre; negative lines are reversed **together with their sample times**.

TE/TR/dwell agree with the Twix files. The sequence specifies a 90° excitation, whereas the Siemens header reports 15°:
the Pulseq file, with a 90° integrated RF pulse, supplies the sequence description.
Its format version is 1.5.1, above the installed PyPulseq reader version 1.5.0, so the calculated result is checked explicitly.
The trajectory describes nominal commands, rather than measured gradient errors.


In [ ]:
sequence = pp.Sequence()
sequence.read(str(SEQUENCE_FILE))
k_adc, _, rf_centres, _, adc_times = sequence.calculate_kspace()
assert k_adc.shape == (3, 35*32) and len(rf_centres) == 1
rf_time = float(rf_centres[0])
kx = k_adc[0].reshape(35,32)[3:].copy()
ky = k_adc[1].reshape(35,32)[3:].copy()
time_s = adc_times.reshape(35,32)[3:].copy()-rf_time
negative = scans['baseline']['negative']
assert np.array_equal(negative, scans['perturbed']['negative'])
for row in range(32):
    assert bool(np.mean(np.diff(kx[row])) < 0) == bool(negative[row])
    if negative[row]:
        kx[row] = kx[row, ::-1]; ky[row] = ky[row, ::-1]; time_s[row] = time_s[row, ::-1]
expected_k = (np.arange(32)-16)/FOV_M
assert np.max(np.abs(kx-expected_k[None,:])) < .001
assert np.max(np.abs(ky-expected_k[:,None])) < .001
TE_EPI = float(sequence.definitions['TE']); TR_EPI = float(sequence.definitions['TR'])
ESP = float(sequence.definitions['EffectiveEchoSpacing'])
assert abs(time_s[16,16]-TE_EPI) < 1e-7
for scan in scans.values():
    assert abs(scan['metadata']['TE_s_header']-TE_EPI) <= 1e-6
    assert abs(scan['metadata']['TR_s_header']-TR_EPI) <= 1e-6
    assert abs(scan['metadata']['dwell_s']-11e-6) < 1e-12
# Exact start of each imaging block: dynamic shim is zero before the first block.
block_time = 0.0; adc_block_starts = []
for block_number in sequence.block_events:
    block = sequence.get_block(block_number)
    if block.adc is not None: adc_block_starts.append(block_time-rf_time)
    block_time += sequence.block_durations[block_number]
line_start_s = np.array(adc_block_starts[3:])
assert len(line_start_s) == 32
assert all(time_s[j].min() >= line_start_s[j] for j in range(32))
print('TE/TR [ms]:', TE_EPI*1000, TR_EPI*1000)
print('ESP/dwell [µs]:', ESP*1e6, 11)
print('Pulseq RF [°]:', sequence.definitions['FlipAngle'], '| header [°]:', scans['baseline']['metadata']['flip_deg_header'])
fig, axes = plt.subplots(1,2,figsize=(10,4))
axes[0].plot(k_adc[0],k_adc[1],'.-',markersize=2,linewidth=.3)
axes[0].set(xlabel='kx [cycles/m]',ylabel='ky [cycles/m]',title='ADC trajectory with navigators'); axes[0].axis('equal')
im=axes[1].imshow(time_s*1000,cmap='viridis'); axes[1].set_title('ADC time from RF [ms] — Cartesian order')
fig.colorbar(im,ax=axes[1]); save_figure(fig,'epi_trajectory_timing.png')

## 6. Build the digital-twin input in physical coordinates

DICOM rows advance towards patient +y and columns towards patient +x.
Twix readout points towards patient +y and phase encoding towards patient −x.
Thus `X_m = y_patient` and `Y_m = −x_patient`; rotating an image alone does not establish its geometry.

The baseline EPI RSS image is interpolated onto the 128 × 128 GRE grid. It provides an **effective transverse amplitude at TE**,
rather than an independent measurement of proton density, T1 or T2. Division by 16 accounts for the 16 simulation points per EPI pixel.
This baseline calibration is not an independent validation of the digital twin.

T2* remains Inf by default: these closely spaced and quantized echoes do not provide a robust relaxation map here.
MATLAB accepts an explicit T2* when a measurement or documented assumption is available.


In [ ]:
assert np.allclose(gre_geometry['orientation'], [1,0,0,0,1,0])
assert np.allclose(gre_geometry['spacing_mm'], [1.5625,1.5625])
row, col = np.meshgrid(np.arange(128),np.arange(128), indexing='ij')
patient_x = gre_geometry['position_mm'][0] + col*gre_geometry['spacing_mm'][1]
patient_y = gre_geometry['position_mm'][1] + row*gre_geometry['spacing_mm'][0]
X_m = patient_y*1e-3
Y_m = -patient_x*1e-3
# EPI grid: row=phase, column=readout, FFT origin at pixel 16.
coordinates = np.array([Y_m/FOV_M*32+16, X_m/FOV_M*32+16])
epi_on_gre_grid = ndimage.map_coordinates(recons['baseline']['rss'],coordinates,order=1,mode='grid-wrap')
simulation_mask = epi_on_gre_grid > 0.03*epi_on_gre_grid.max()
m0_proxy = np.where(simulation_mask, epi_on_gre_grid/16, 0)
quality_fraction = np.count_nonzero(quality & simulation_mask)/np.count_nonzero(simulation_mask)
print('Fraction of simulation support with reliable measured B0:', f'{quality_fraction:.1%}')
fig, axes = plt.subplots(1,3,figsize=(11,4))
for ax,a,title in zip(axes,[recons['baseline']['rss'],epi_on_gre_grid,quality & simulation_mask],
                      ['Baseline EPI [phase, readout]','Same amplitude on the GRE grid','Reliable measured B0 ∩ twin support']):
    ax.imshow(a,cmap='gray'); ax.set_title(title); ax.axis('off')
save_figure(fig,'twin_input_geometry.png')

## 7. Export to MATLAB

`experiment_input.mat` contains measured images, the masked B0 map, its explicitly hypothetical extension,
physical geometry, ADC trajectory and model input. It contains no invented ACDC calibration.

The MATLAB model evolves transverse precession between excitation and ADC:

$$s(t)=\sum_r m_{TE}(r)\exp[-(t-TE)/T_2^*(r)]\exp[-2\pi i(k_xX+k_yY+B_0t)].$$

Uniform reception is retained as an approximation of the historical model, whereas acquired images combine 32 receive channels using RSS.
The zero-B0 image is a reference scenario. The perturbed simulation uses the total GRE B0 and its extension;
it does not validate the A22 term alone while baseline B0 remains unmeasured.

These files belong to the historical virtual analysis. Use the new step-by-step notebook for the measured-coverage experiment input and controller export.


In [ ]:
assumptions = [
    'Baseline scan has no added A22 perturbation; its intrinsic B0 was not measured.',
    'GRE B0 is total field in the perturbed condition; scanner shim settings also differ between scans.',
    'B0 outside the quality mask is nearest-neighbour extrapolation, not a measured field.',
    'Spatial phase unwrapping does not determine absolute frequency branch or scanner/simulator sign.',
    'M0 proxy is derived from baseline RSS at TE; it is not measured proton density.',
    'Uniform receive sensitivity approximates the measured 32-channel RSS combination.',
    'T2star is Inf by default; RF shape, slice profile and TR steady state are not simulated.',
    'Pulseq nominal ADC trajectory is used; scanner gradient imperfections are not measured.',
    'Dynamic shim currents are defined as zero before the first imaging block.',
    'ACDC coils were absent during the scans; library maps are imported separately for a virtual coil placement.',
]
provenance = {
    'baseline_file': baseline_file.name, 'perturbed_file': perturbed_file.name,
    'sequence_file': SEQUENCE_FILE.name,
    'sequence_sha256': hashlib.sha256(SEQUENCE_FILE.read_bytes()).hexdigest(),
    'TE_GRE_s': TE_GRE, 'B0_sign': B0_SIGN,
    'MIN_MAGNITUDE': MIN_MAGNITUDE, 'MAX_PHASE_RESIDUAL': MAX_PHASE_RESIDUAL,
    'baseline_transmitter_Hz': scans['baseline']['metadata']['transmitter_Hz'],
    'perturbed_transmitter_Hz': scans['perturbed']['metadata']['transmitter_Hz'],
    'navigator_correction': 'Three +,-,+ navigators; per-coil weighted affine phase fit; negative lines corrected.',
}
experiment = {
    'images': {'baseline':recons['baseline']['rss'], 'perturbed':recons['perturbed']['rss'],
               'baseline_coils':recons['baseline']['coil_images'], 'perturbed_coils':recons['perturbed']['coil_images']},
    'field': {'b0Hz':b0_for_simulation, 'measured_b0Hz':b0_measured,
              'mask':quality, 'simulation_mask':simulation_mask, 'phase_residual_rad':phase_residual},
    'geometry': {'X_m':X_m, 'Y_m':Y_m, 'position_mm':gre_geometry['position_mm'],
                 'orientation':gre_geometry['orientation'], 'spacing_mm':gre_geometry['spacing_mm'],
                 'axes':'X=patient+y (readout); Y=patient-x (phase)'},
    'acquisition': {'kx_mInv':kx, 'ky_mInv':ky, 'time_s':time_s,
                    'line_index':np.repeat(np.arange(1,33)[:,None],32,axis=1),
                    'line_start_s':line_start_s[:,None], 'polarity':np.where(negative,-1,1),
                    'dwell_s':11e-6, 'TE_s':TE_EPI, 'TR_s':TR_EPI, 'echo_spacing_s':ESP,
                    'rf_flip_deg':90.0, 'header_flip_deg':scans['baseline']['metadata']['flip_deg_header'],
                    'FOV_m':FOV_M, 'matrix_size':32},
    'm0_proxy':m0_proxy, 'relaxation':{'T2star_s':np.inf},
    'assumptions':np.array(assumptions,dtype=object), 'provenance':provenance,
    'optimization_ready':False,
}
savemat(PREPARED/'experiment_input.mat', {'experiment':experiment},do_compression=True,long_field_names=True)
np.savez_compressed(PREPARED/'reconstruction_arrays.npz',
    baseline=recons['baseline']['rss'],perturbed=recons['perturbed']['rss'],
    b0_measured=b0_measured,b0_for_simulation=b0_for_simulation,quality_mask=quality,
    phase_residual=phase_residual,kx=kx,ky=ky,time_s=time_s,m0_proxy=m0_proxy,X_m=X_m,Y_m=Y_m)
summary = {
    'image_matrix':[32,32], 'object_grid':[128,128], 'reliable_B0_voxels':int(quality.sum()),
    'simulation_support_voxels':int(simulation_mask.sum()), 'reliable_B0_support_fraction':float(quality_fraction),
    'TE_EPI_s':TE_EPI, 'TR_EPI_s':TR_EPI, 'echo_spacing_s':ESP,
    'TE_GRE_s':TE_GRE.tolist(), 'assumptions':assumptions,
    'sequence_sha256':provenance['sequence_sha256'], 'optimization_ready':False,
}
(PREPARED/'preparation_summary.json').write_text(json.dumps(summary,indent=2,ensure_ascii=False)+'\n')
print('MATLAB export:', PREPARED/'experiment_input.mat')
print('ACDC optimization ready:', experiment['optimization_ready'])

## 8. Continue in MATLAB

From the project directory:

```matlab
addpath('Experiment');
[run_dir, report] = run_experiment_digital_twin();
```

The four panels share one scale and **one gain**, fitted on baseline and retained for the perturbed condition.
MATLAB exports `.fig`, `.png`, data and metrics. Error values describe this conditional approximation.
For virtual optimization with the supplied 3 T responses, follow `CALIBRATION_ACDC.md` and `OPTIMIZATION_SPEED.md`:

```matlab
[run_dir, summary] = run_experiment_virtual_acdc('cpu', 'MaxIterations', 3);
```

The covered input is prepared separately. The default maximum is 600 iterations, with 1 A per channel and absolute-current sum <30 A per line.

## Method and reader references

- [twixtools: MDH, image and PHASCOR reading](https://github.com/pehses/twixtools)
- [Pulseq: open sequence format](https://pulseq.github.io/)
- [PyPulseq: trajectory extraction](https://github.com/imr-framework/pypulseq)
- [SPM FieldMap: phase difference and conversion to Hz](https://www.fil.ion.ucl.ac.uk/spm/docs/manual/FieldMap/fieldmap/)
- [scikit-image: phase unwrapping](https://scikit-image.org/docs/stable/auto_examples/filters/plot_phase_unwrap.html)


## 9. MATLAB result for these historical data

The comparison below was calculated with `run_experiment_digital_twin.m`.
Relative NRMSE is approximately **18.6%** on baseline and **73.6%** on the perturbed scan.
Baseline defines the digital-twin amplitude; agreement there is not an independent validation.
The perturbed twin remains an approximation: baseline B0 is unknown, much of the simulated field is extended beyond measured coverage, and reception is uniform.

For a stronger physical comparison, acquire baseline B0 in the same scanner state, increase GRE signal and mask coverage,
then characterize amplitude and receive sensitivities more accurately. The 3 T ACDC maps already permit a virtual study with their assumed placement explicitly recorded.


In [ ]:
from IPython.display import Image
report_path = PREPARED/'digital_twin_summary.json'
comparison_path = FIG/'digital_twin_comparison.png'
if report_path.exists() and comparison_path.exists():
    comparison_summary = json.loads(report_path.read_text())
    print('Baseline NRMSE:', f"{comparison_summary['baseline_nrmse']:.1%}")
    print('Perturbed NRMSE:', f"{comparison_summary['perturbed_nrmse']:.1%}")
    display(Image(filename=str(comparison_path)))
else:
    print('Run run_experiment_digital_twin.m in MATLAB to generate the comparison.')